# Freezing Point Depression using a Beckmann Cryoscopic Apparatus

<div>
  <center> <img width="500" src="https://github.com/act-cms/freezing-point-depression-molar-mass-determination/blob/dev/02_freezing-point-depression-analysis/Figures/Beckman_Cryoscopic.png?raw=True"> </center>
</div>

The experimental set-up is a Beckmann cryoscopic apparatus, named for the physical chemist Ernst Otto Beckmann (1853-1923) who invented the technique in 1887. The apparatus consists of a nested set of three containers. The inner-most container is a test tube into which the solvent and solute are placed and monitored with a thermometer as the temperature is lowered. The outer-most container is a large glass vessel in which the cooling medium is placed. In this experiment, the cooling medium is ice-water with salt. The inner-most container is separated from the outermost contained by a buffer container with an air-gap. The reason for this middle container is to ensure that the temperature is uniform in the inner-most container during the cooling. Both the inner-most and outer-most containers have a manual stirring rod used for maintaining temperature uniformity.

This particular experiment will determine the molar mass of p-dibromobenzene by measuring how much the freezing point is lowered by the addition of a measured amount of an unknown solute to a weighed amount of cyclohexane. This technique should allow you to determine the molar mass of a substance $\le$500 g/mol to within 3 to 5\%.

Temperature will be monitored with a Vernier Surface Temperature Sensor (STS-BTA). The Vernier-BTA temperature sensor has a 95\% confidence interval of 0.03$^\circ$C.

:::{admonition} Overview
- Data Filtering and Slicing.
- Data Smoothing.
- Numerical differentiation.
- Constructing Python functions to repeat a task.
:::

In [ ]:
# Import standard packages
import numpy as np # Import numerical analysis
import os,sys,re # Import regex
import pandas as pd # DataFrame analysis
%pip install openpyxl --quiet

# SciPy packages
from scipy import interpolate
import scipy.optimize
from scipy import stats
from scipy.signal import savgol_filter #Savitzky–Golay Filter

# Plotting
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 16})

# Interactive Plots
import ipywidgets as widgets
from ipywidgets import interact

# Insert a progress bar to show the progress of the script
from tqdm.notebook import tqdm, tnrange, trange

#Quizzes
from IPython.display import display, HTML
import time

%pip install "jupyterquiz" --quiet
from jupyterquiz import display_quiz

import sys
sys.path.append('..')  # Add parent directory to path
import quiz_utils      # Now you can import it directly

## Thermodynamics Background

When a solute is dissolved in a solvent, it **lowers the solvent’s freezing point**. This phenomenon, known as **freezing point depression**, is one of several *colligative properties*, which depend only on the number of solute particles, not their chemical identity.

At equilibrium, the chemical potentials of the solid and liquid phases are equal

$$
\mu_A(\text{s}) = \mu_A(\text{l})
$$

When a non-volatile solute is added, the liquid phase’s chemical potential decreases because mixing increases entropy. To re-establish equilibrium between solid and liquid, the system must cool to a lower temperature before the solid and liquid can coexist again.

---

### Deriving the Freezing Point Depression

Starting from the equilibrium condition and applying a first-order approximation, one can derive:

$$
\Delta T = -\frac{\chi_B R T_{\text{fus}}^{*2}}{\Delta H_{\text{fus}}}
$$

where:

* $\chi_B$: mole fraction of solute
* $R$: gas constant
* $T_{\text{fus}}^*$: freezing point of the pure solvent
* $\Delta H_{\text{fus}}$: molar enthalpy of fusion of the solvent

The constant term in front of $\chi_B$ is defined as the **freezing-point depression constant** for a given solvent:

$$
K_f = \frac{R T_{\text{fus}}^{*2}}{\Delta H_{\text{fus}}}
$$

Thus,

$$
\Delta T = -K_f \chi_B
$$

This shows that the freezing point depression $\Delta T$ is directly proportional to the solute concentration.

---

### Converting to Experimentally Useful Units

In the lab, we often express solute concentration in **molality**, $b_B$:

$$
b_B = \frac{n_B}{m_A} = \frac{\chi_B}{M_A} 
$$

where:

* $n_B$: moles of solute
* $m_A$: mass of solvent (kg)
* $M_A$: molar mass of the solvent

Substituting this into the previous equation gives:

$$
\Delta T = -K_f M_A b_B = -K_f' b_B
$$

where $K_f' = K_f M_A$ is the **freezing-point depression constant** in units of K\*kg/mol.

---

### Experimental Meaning

This equation means that the **change in freezing point** is proportional to the **molality of the solute**:

$$
\Delta T \propto b_B
$$

In practice:

* The **slope** of a plot of $\Delta T$ vs. $b_B$ gives $K_f'$, the solvent constant.
* If $K_f'$ is known, measuring $\Delta T$ allows the **molar mass of an unknown solute** to be determined.

The data analysis in this notebook (finding the intersection of cooling curve segments) provides the **experimental freezing temperature** needed to calculate $\Delta T$.

## Data Analysis Overview

When a substance cools, its temperature decreases over time until it reaches the freezing point, where the temperature levels off as liquid begins to solidify. In an ideal pure substance, this plateau is flat, showing that the temperature stays constant while freezing occurs. However, in practice (especially for mixtures), the curve isn’t perfectly flat. To accurately determine the freezing point, we analyze the cooling curve mathematically.

As the liquid cools:

* The first region (before freezing starts) shows a roughly linear decrease in temperature with time.

* The second region (after freezing begins) also follows a nearly linear trend but with a shallower slope, since heat is being released as the liquid solidifies.

By fitting straight lines to each of these regions, we can find their intersection point, which corresponds to the freezing temperature $(T_{fusion})$ of the solution.

<div>
  <center> <img width="500" src="https://github.com/act-cms/freezing-point-depression-molar-mass-determination/blob/dev/02_freezing-point-depression-analysis/Figures/SampleData_Plots.png?raw=True"> </center>
</div>

In the image above, the cooling curve for a pure substance looks like the left curve while the right curve is one for a mixture. The pure substance should yield a flat line after supercooling (minimum) whereas the mixture shows a sharp decrease.

We will use a combination of **data smoothing** to smooth the noisy temperature data as well as **linear algebra** to determine the freezing temperature for a pure solution of cyclohexane and two mixtures of cyclohexane with p-dibromobenzene. Once the freezing temperatures are obtained, we can use Equation 6 to **determine the molar mass of p-dibromobenzene**.

# Part 1. Data Smoothing

In [ ]:
# Import your data here
data_pure =
data_mix1 =
data_mix2 =

:::{warning} ✏️ Exercise
:icon: false
For the pure cyclohexane data (`data_pure`), do the following:

- Use the rolling average and the Savitzky–Golay method to create smoother data.
- Use the smoothed data to filter the data to isolate pre- and post-supercooling linear lines.
- Find the intersection between the two lines to determine the freezing point ($T_{fus}$).
:::

## 1.1 Moving averages

A moving average calculates the average over a specified range and then propagates that average as it increases in index.
Convert your data that is in a dataframe to the moving average.

For example,

```python
x = pd.DataFrame([0,0,0,1,1,1,2,2,2])
y = x.rolling(3).mean().dropna()
print(y)

# Output from moving average
2 0.000
3 0.333
4 0.667
5 1.000
6 1.333
7 1.667
8 2.000
```

In [ ]:
# Calculate the moving average of data_pure(Note: 25-30 is optimal)
# Create a scratch cell (Ctrl+Alt+N) to visualize the roll_pure dataframe
roll_pure =

# Create the plt.figure()
fig = plt.figure()

# These settings will help the contrast for visualizing the data
# Plot the raw data (use alpha=0.5)

# Plot the moving average (use lw=2 for a thicker line)
# You will need to use the roll_pure DataFrame to plot the rolling average

# Label the axes
plt.show()

:::{danger} ❓ Question 1
:icon: false
What happens as you change the size of the rolling average?

- How does the curve change?
- What happens to the DataFrame?
:::{warning} 📝 Student Response
:icon: false
To record your response, double click the cell and type your answer after the `>` below!
> _Type your answer here!_
::::::

## 1.2 Savitzy-Golay Smoothing

<center> <img width="800"
  src="https://github.com/act-cms/freezing-point-depression-molar-mass-determination/blob/dev/02_freezing-point-depression-analysis/Figures/SGfilter_Wiki.gif?raw=True"> </center>
gif designed By <a href="//commons.wikimedia.org/w/index.php?title=User:MothNik&amp;action=edit&amp;redlink=1" class="new" title="User:MothNik (page does not exist)">MothNik</a> - <span class="int-own-work" lang="en">Own work</span>, <a href="http://creativecommons.org/publicdomain/zero/1.0/deed.en" title="Creative Commons Zero, Public Domain Dedication">CC0</a>, <a href="https://commons.wikimedia.org/w/index.php?curid=164353357">Link</a>



:::{hint} 🐍 Savitzky–Golay Smoothing: How?
:icon: false

When collecting experimental data, such as the temperature of a solution over time as it freezes, random fluctuations (noise) can make it difficult to clearly see key features like the freezing point plateau. The **Savitzky–Golay filter** helps reduce this noise while preserving the overall shape of the curve. It does this by fitting a polynomial to small sections of the data and using that to estimate each central value.

**How it works:**

- Pick a small "window size" (e.g., 5 or 7 points) around each data point.
- Fit a low-degree polynomial (e.g., quadratic or cubic) to those points.
- Use the fitted polynomial to smooth the value at the center of the window.

The fitted polynomial has the general form:

$$y(x) = a_0 + a_1 x + a_2 x^2 + \cdots + a_k x^k$$

To find the best fit, the filter minimizes the sum of squared errors:

$$\sum_{j=-m}^{m} \left[ y_{i+j} - \left( a_0 + a_1 x_{i+j} + \cdots + a_k x_{i+j}^k \right) \right]^2$$

After the polynomial is fit, the smoothed value at the center of the window is computed using:

$$\tilde{y}_i = \sum_{j=-m}^{m} c_j \cdot y_{i+j}$$

where $c_j$ are precomputed weights based on the window size and polynomial order.

**Why are we using this filter?**

- It keeps the curve smooth without blurring important features.
- It's better than a moving average for curves with distinct changes.
:::

:::{hint} 🐍 Note
:icon: false
The syntax to use the Savitzy-Golay method is as follows:

```python
savgol_filter(data, window_size, polynomial_order)
```

For example,
```python
savgol_filter(data1['Temperature'], 30, 2)
```
applies a quadratic fitting (polynomial order 2) to a window size of 30 data points to the column in your DataFrame (named data1) called Temperature.

**Note: The window size should be between 50-101 data points for your data**


In [ ]:
# Try different window sizes and polynomial orders
# Record your final choices in your notebook
smooth_pure =

fig = plt.figure()


# Label the axes


plt.show()

In [ ]:
# Plot the raw data, the rolling average, and the smoothed data onto one plot
fig = plt.figure()


# Label the axes

plt.show()

:::{danger} ❓ Question 2
:icon: false
Are there any noticeable differences between the two smoothing methods?
:::{warning} 📝 Student Response
:icon: false
To record your response, double click the cell and type your answer after the `>` below!
> _Type your answer here!_
::::::

# Part 2. Determining $T_{fus}$


:::{warning} ✏️ Exercise
:icon: false
$T_{fus}$ is determined to be the intersection of the cooling lines pre- and post-supercooling. Play with the interactive plot below and see what you can gain from the analysis.

Be intentional about which data points you are selecting and think about the physical and/or mathematical reason for why you are selecting those data points.
:::

## 2.1 Interactive Plot

In [ ]:
%run Tfus_interactive.py

:::{danger} ❓ Question 3
:icon: false
- What did you consider in selecting your pre- and post-supercooling lines?
- How close did you get to the actual molar mass using the sample data?
- What were your observations between the original plot and $dT/dt$?
:::{warning} 📝 Student Response
:icon: false
To record your response, double click the cell and type your answer after the `>` below!
>  Note to Instructors: Set t1=0, t2=66.8, t3=109.6, t4=227. This will return a calculated molar mass of 235.906 g/mol. The students should be able to see how the derivatives are relatively constant for pre- and post-supercooling and select the points that correspond to flat derivatives accordingly.
>
> From a physical perspective, the two distinct regions of a cooling curve correspond to different physical processes occurring in the sample. Before freezing, the entire sample is liquid, and it's temperature decreases steadily as heat is removed $(q=mC_p\Delta T)$. During freezing, the phase changes, and part of the heat removed no longer lowers the temperature but goes towards the energy required to convert liquid to solid. This is represented as the chemical potentials of liquid and solid being equal $(\mu_l=\mu_s)$.
>
> In the liquid region $\mu_l > \mu_s$. At the freezing point, $\mu_l=\mu_s$. Once both phases coexist, further removal of heat mainly shifts the phase boundary rather than decreasing the temperature, hence the reduced slope.
::::::


## 2.1 Build the components

**We are going to build the individual components and then place that to interactively determine $T_{fus}$ for our data.**

First, we need to figure out the derivative function that was plotted on the right. There are a few functions that accomplish this. We will use the numpy difference (np.diff) function of the smoothed data, which calculates the difference between adjacent indices in a row/column (specify axis=0 or axis=1), and then divide by dt (0.2) to plot the derivative numerically. There are SciPy functions that also compute the derivative of a function.

```python
np.diff()
```

Because the first difference is given by ``out[i] = a[i+1] - a[i]`` along the given axis, we have to adjust the time since there will be one less data point. This can be done by slicing the data using [1:] when using the savgol_filter. If using the rolling average, you will need to account for the size of your window.

```python
time=np.array(data_pure['Time'][1:]) #Assuming data_pure is a dataframe -- convert to an array
time=data_pure[1:] #Assuming data_pure is an array
```


In [ ]:
# Use the np.diff function on the pure data to calculate dT
# Make this a DataFrame with a labeled column called Temperature
diff_pure=

# Adjust the x scale for the updated difference data
time=

#Calculate the derivative dT/dt by dividing dT by 0.2 (or dt)
dT_dt=

# Generate a plot of the dT/dt as a function of time

plt.ylabel(r'dT/dt ($\degree$C/s)')
plt.xlabel('Time (s)')
plt.show()

**Does the data look noisy?**

Let's use the savgol_filter function to smooth the derivative data.

In [ ]:
# Use the savgol_filter to smooth the derivative data (remember to divide by dt here)
diff_pure_savgol=

# Visualize the data to highlight the differences
# Use a lighter color (alpha=0.3) for the raw data
# Use a darker color for the smoothed data

plt.ylabel(r'dT/dt ($\degree$C/s)')
plt.xlabel('Time (s)')
plt.show()

:::{warning} ✏️ Exercise
:icon: false
For a manual selection of data points, you can select certain data points based on criteria you define. Conditional statements can filter data if they meet certain criteria, such as temperature and time.

Filter the data via a boolean conditional statement for time:

- One should be less than a certain value (e.g., `t < 30`) for the pre-supercooling line.
- One should be greater than another value (e.g., `t > 100`) for the post-supercooling line.
:::

In [ ]:
# Use filters (time < 30) to try and isolate the correct data points
diff_pure_filter_pre  =
diff_pure_filter_post =

# Remember to apply the same conditional to the time
time_filter_pre  =
time_filter_post =

# Then plot the filtered values

# Place your code here for the full derivative data
# Use gray for the raw data with alpha=0.3
# Use black for the smoothed data


# Place your code here for the filtered derivative data
# Use red for pre-supercooling ('red', or 'r')


# Use blue for post-supercooling ('blue', or 'b')


# Here are the axes labels
plt.ylabel(r'dT/dt ($\degree$C/s)')
plt.xlabel('Time (s)')
plt.show()

:::{hint} 🐍 Note
:icon: false
**Subplots within matplotlib**

When you want to compare data, like in this case of temperature vs. time AND $dT/dt$ vs. time, it is best to show them together using subplots. This will allow you to export both figures as one image, allowing for clear, organized visualization.

`plt.subplots()` creates a grid of smaller plots (axes) within one figure. You can control how many rows and columns you want.
:::

:::{hint} 🐍 Note
:icon: false
Make sure to adjust the `'Time'` column to match your dataset column label.
:::

In [ ]:
# Apply the same filters to the original data (note that the syntax changes a little)

fig, [ax1, ax2] = plt.subplots(nrows=1, ncols=2, figsize=(18,6),sharex=True)

ax1.plot(data_pure['Time'],data_pure['Temperature'],'gray',alpha=0.3,label='Raw Data')
ax1.plot(data_pure['Time'],smooth_pure,'k-',label='Smoothed Data')

ax1.plot(data_pure['Time'][data_pure['Time']<30],data_pure['Temperature'][data_pure['Time']<30],'red',alpha=0.3)
ax1.plot(data_pure['Time'][data_pure['Time']<30],smooth_pure[data_pure['Time']<30],'r',alpha=0.75)

ax1.plot(data_pure['Time'][data_pure['Time']>100],data_pure['Temperature'][data_pure['Time']>100],'blue',alpha=0.3)
ax1.plot(data_pure['Time'][data_pure['Time']>100],smooth_pure[data_pure['Time']>100],'blue',alpha=0.75)

# Label the axes
ax1.set_xlabel('Time (s)')
ax1.set_ylabel(r'Temperature ($^\degree$C)')
ax1.legend(loc=0)

# Copy your code for the derivative plot here
# Change plt to ax2



# Here are the axes labels
ax2.set_ylabel(r'dT/dt ($\degree$C/s)')
ax2.set_xlabel('Time (s)')

#Export the figure
plt.show()
fig.savefig('Plot_with_Derivative.png',dpi=300,bbox_inches='tight')

:::{hint} 🐍 Cramer's Rule
:icon:false
Now that you have a method to filter the data, we can construct best fit lines and then use Cramer's Rule to determine $T_{fusion}$ as the intersection between the two best fit lines.

**Cramer's Rule** provides a way to solve a system of linear equations using determinants.
For two equations in two unknowns,

$$
\begin{cases}
m_1x + a_1y = b_1 \\
m_2x + a_2y = b_2
\end{cases}
$$

we can represent the system in matrix form as

$$
\begin{bmatrix}
m_1 & a_1 \\
m_2 & a_2
\end{bmatrix}\begin{bmatrix}
x \\
y
\end{bmatrix}=\begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}
$$


Cramer's Rule states that:

$$
x = \frac{\det\bigg(\begin{bmatrix}
b_1 & a_1 \\
b_2 & a_2
\end{bmatrix}\bigg)}{\det\bigg(\begin{bmatrix}
m_1 & a_1 \\
m_2 & a_2
\end{bmatrix}\bigg)} = \dfrac{b_1a_2-b_2a_1}{m_1a_2-m_2a_1}, \quad y = \frac{\det\bigg(\begin{bmatrix}
m_1 & b_1 \\
m_2 & b_2
\end{bmatrix}\bigg)}{\det\bigg(\begin{bmatrix}
m_1 & a_1 \\
m_2 & a_2
\end{bmatrix}\bigg)} = \dfrac{m_1b_2-m_2b_1}{m_1a_2-m_2a_1}
$$


The **intersection point** $(x, y)$ of two lines given in standard form can thus be found by applying Cramer's Rule *as long as* $m_1a_2-m_2a_1 \neq 0$, i.e., the lines are not parallel.

To apply this to our lines, since we are solving the lines in the form of $y=mx+b$ with ```scipy.stats.linregress()```, we will get the following matrix representation

$$
\begin{bmatrix}
-m_1 & 1 \\
-m_2 & 1
\end{bmatrix}\begin{bmatrix}
x \\
y
\end{bmatrix}
=\begin{bmatrix}
b_1 \\
b_2
\end{bmatrix}
$$

where $m_1$ and $b_1$ are the slope and intercept of the first line, respectively, and $m_2$ and $b_2$ are the slope and intercept of the second line.

Therefore,
$$
x =  \dfrac{b_1-b_2}{m_1-m_2}, \quad y = \dfrac{m_1b_2-m_2b_1}{m_1-m_2} = m_1x_{int} + b_1
$$

The following code can be used for Cramer's Rule
```python
# Cramer's Rule for intersection
D = (-slope1 + slope2) # Determinant
if D != 0:
  x_int = (intercept1 - intercept2) / D
  y_int = slope1*x_int + intercept1
else:
  x_int=np.nan
  y_int=np.nan
```
:::

In [ ]:
#Generate the best fit lines based on your filtered data
best_fit_pre=
best_fit_post=

slope1=
intercept1=

slope2=
intercept2=

# Copy Cramer's Rule code here


print(r'T_fus = '+f'{y_int:.3f}°C')

## 2.2 Construct the interactive figure

You will provide the data sets and then interpret the interactive function to determine $T_{fusion}$.

In [ ]:
data=       # The respective DataFrame (data_pure, data_mix1, data_mix2)
time=       # Raw data of time for your respective DataFrame as an array
temp_data=  # Raw data for temperature for your respective DataFrame as an array

temperature =    # Apply the savgol_filter to the raw temperature data

dT_dt= # Calculate dT/dt
dT_dt_savgol= # Apply the savgol_filter() function
dT_time= # Use the time based on the differences

:::{hint} 🐍 Note
:icon: false
- The code below will generate a 300 dpi image named **Interactive_Tfusion.png** when adjusting the sliders that refreshes each time you use the sliders.
- If your column headers are not labeled **'Time'** or **'Temperature'**, then change them to match accordingly. Use the find and replace tool to change the column names.
:::

::: {danger} 
DO NOT CHANGE except to update the column names for Time and Temperature
:::

In [ ]:
# DO NOT CHANGE except to update the column names for Time and Temperature

# --- Define the interactive plotting function ---
def plot_Tfusion(t1_start=0, t1_end=time[20], t2_start=time[-100], t2_end=time[-1]):
  ''' Generates the interactive subplots for the cooling curve and its derivative'''
  fig, [ax1, ax2] = plt.subplots(nrows=1, ncols=2, figsize=(18,6),sharex=True)
  ax1.plot(time,data['Temperature'], 'gray', alpha=0.3, label='Raw Data')
  ax1.plot(time,temperature, 'black', alpha=0.7, label='Smooth Data')

  # --- Pre-supercooling ---
  # Use the smooth data for the regression
  seg1 = data[(data['Time'] >= t1_start) & (data['Time'] <= t1_end)].copy()
  seg1A = seg1.copy()
  seg1A['Temperature'] = data['Temperature'][(data['Time'] >= t1_start) & (data['Time'] <= t1_end)]
  seg1['Temperature'] = temperature[(data['Time'] >= t1_start) & (data['Time'] <= t1_end)]
  if len(seg1) >= 2:
      slope1, intercept1, r_value1, p_value1, std_err1 = scipy.stats.linregress(seg1['Time'], seg1['Temperature'])
      # Extend the line across the full time range
      ax1.plot(data['Time'], slope1*data['Time'] + intercept1, 'r--', label=f'Fit 1: T={slope1:.4f}t+{intercept1:.2f}')

      ax1.plot(seg1['Time'], seg1['Temperature'], 'r', alpha=0.75,lw=2)
      ax1.plot(seg1A['Time'], seg1A['Temperature'], 'r', alpha=0.35)

  # --- Post-supercooling ---
  # Use the smooth data for the regression
  seg2 = data[(data['Time'] >= t2_start) & (data['Time'] <= t2_end)].copy()
  seg2A = seg2.copy()
  seg2A['Temperature'] = data['Temperature'][(data['Time'] >= t2_start) & (data['Time'] <= t2_end)]
  seg2['Temperature'] = temperature[(data['Time'] >= t2_start) & (data['Time'] <= t2_end)]

  if len(seg2) >= 2:
      slope2, intercept2, r_value2, p_value2, std_err2 = scipy.stats.linregress(seg2['Time'], seg2['Temperature'])
      # Extend the line across the full time range
      ax1.plot(data['Time'], slope2*data['Time'] + intercept2, 'b--', label=f'Fit 2: T={slope2:.4f}t+{intercept2:.2f}')
      ax1.plot(seg2['Time'], seg2['Temperature'], 'b', alpha=0.75,lw=2)
      ax1.plot(seg2A['Time'], seg2A['Temperature'], 'b', alpha=0.35)

      # Cramer's Rule
      D = (-slope1 + slope2)
      if D != 0:
          x_int = (intercept1 - intercept2) / D
          y_int = slope1*x_int + intercept1
          ax1.scatter(x_int, y_int, color='k', s=60, zorder=5, label=(r'T$_{fus}$ = '+f'{y_int:.3f}°C'))
      else:
          x_int = np.nan
          y_int = np.nan

  ax1.set_xlabel('Time (s)')
  ax1.set_ylabel(r'Temperature ($\degree$C)')
  ax1.legend()
  ax1.grid(True)
  ax1.set_ylim(min(data['Temperature'])-0.1,max(data['Temperature'])+0.1)

  ax2.plot(dT_time,dT_dt,'gray',alpha=0.3,label='Raw dT/dt')
  ax2.plot(dT_time,dT_dt_savgol,'black', alpha=0.7, label='Smoothed dT/dt')

  # Highlight the selected regions in the difference plot
  diff_seg1_indices = (dT_time >= t1_start) & (dT_time <= t1_end)
  ax2.plot(dT_time[diff_seg1_indices], dT_dt[diff_seg1_indices], 'r', alpha=0.35, lw=2)
  ax2.plot(dT_time[diff_seg1_indices], dT_dt_savgol[diff_seg1_indices], 'r', alpha=0.75)

  diff_seg2_indices = (dT_time >= t2_start) & (dT_time <= t2_end)
  ax2.plot(dT_time[diff_seg2_indices], dT_dt[diff_seg2_indices], 'b', alpha=0.35, lw=2)
  ax2.plot(dT_time[diff_seg2_indices], dT_dt_savgol[diff_seg2_indices], 'b', alpha=0.75)
  ax2.set_xlabel('Time (s)')
  ax2.set_ylabel(r'dT/dt ($\degree$C/s)')
  ax2.grid(True)

  ax2.legend()
  plt.show()
  fig.savefig('Interactive_Tfusion.png',dpi=300,bbox_inches='tight')

# Create four sliders (units of seconds)
interact(
    plot_Tfusion,
    t1_start=widgets.FloatSlider(value=0, min=0, max=time[-1], step=0.2, description='t1 (s)'),
    t1_end=widgets.FloatSlider(value=time[20], min=0, max=time[-1], step=0.2, description='t2 (s)'),
    t2_start=widgets.FloatSlider(value=time[-100], min=0, max=time[-1], step=0.2, description='t3 (s)'),
    t2_end=widgets.FloatSlider(value=time[-1], min=0, max=time[-1], step=0.2, description='t4 (s)')
);

:::{danger} ❓ Question 4
:icon: false
Interpret the interactive plot function (`plot_Tfusion`). Discuss with your lab partners what parts are similar/different to the code we generated for the components. What data slicing filters were applied?
:::{warning} 📝 Student Response
:icon: false
To record your response, double click the cell and type your answer after the `>` below!
> _Type your answer here!_
::::::

# Part 3. Process the rest of your data



## 3.1 Make your own function

### ***Wouldn’t it be nice if you didn’t have to copy and paste the same code over and over?***

**Python functions** are extremely useful for handling **repetitive tasks** because they let you write a block of code once and reuse it whenever needed. Instead of copying and pasting the same code multiple times, **you can define a function that performs the task and simply call it by name**. This saves time, reduces typing errors, and makes your code cleaner and easier to maintain.

:::{warning} ✏️ Exercise
:icon: false
- Create a function that uses a DataFrame (`data_pure`, `data_mix1`, `data_mix2`) as the input.
- Combine the code in the previous part to generate the interactive figure to determine $T_{fusion}$.
  - You can include as many variables as you need.
  - This way, you don't have to repeat the same calculations and change variables every single time you introduce a new dataset.
  - Calculate the respective $T_{fus}$, from which you can determine the molar mass of p-dibromobenzene.
:::

In [ ]:
# The output should be a generalized version of the interactive double figure that can be applied to any data set

def determine_Tfus():


:::{hint} 🐍 Note
:icon: false
**How should you make a function to automate a task you've already done once?**

- You can copy and paste your code from either section above that completed the task for one set of data.
- Find the variables that you know you will need to replace between runs to create the function.
- Follow the Python notation to create a function of multiple variables. The goal of this Python function is to be able to repeat the task for whatever set of data you want it to.

This will allow you to create a function to automate a task and not have to worry about copying multiple blocks of code depending on your data.
:::

## 3.2 Try out your function

Use your newly created ```determine_Tfus()``` function to find $T_{fusion}$ for data_mix1 and data_mix2.

In [ ]:
# Call your function and work with the data from the first mixture (data_mix1)


In [ ]:
# Call your function to work with the data from the second mixture (data_mix2)


# Part 4. Calculate the molar mass of p-dibromobenzene

Using your values for $T_{fusion}$, you will calculate the molar mass of p-dibromobenzene (pDBrB) by computing the following from Equations 5 and 6.

$$
b_B = \frac{n_B}{m_A} = \frac{\chi_B}{M_A} \tag{5}
$$

$$
\Delta T = -K_f M_A b_B = -K_f' b_B\tag{6}
$$
The value of $K_f'$ for cyclohexane is -20.459 K\*kg/mol.

To solve for molar mass of p-dibromobenzene (pDBrB),
$$
MW_{pDBrB} = \dfrac{m_{pDBrB}}{b_B*m_{cyclohexane}}
$$

In [ ]:
# For Mixture 1
m_pDBrB=
m_cyclo= # mass in kg
T_fus=
T_0=
Kf1=
Delta_T=
bB=
MW_pDBrB =

print(f"Calculated Molar Mass: {MW_pDBrB:.3f} g/mol")
print("Actual Molar Mass: 235.906 g/mol")

In [ ]:
# For Mixture 2
m_pDBrB=
m_cyclo= # mass in kg
T_fus=
T_0=
Kf1=
Delta_T=
bB=
MW_pDBrB =

print(f"Calculated Molar Mass: {MW_pDBrB:.3f} g/mol")
print("Actual Molar Mass: 235.906 g/mol")

In [ ]:
# Simplified Error Propagation
Delta_T_95CI=
bB_95CI=

# For the mix1 solution (adding 0.2-0.3 g pDBrB to cyclohexane)
MW_pDBrB_mix1_95CI=

# For the mix2 solution (adding another 0.2-0.3 g pDBrB to the mixture)
MW_pDBrB_mix2_95CI=

print(f"Calculated Molar Mass: {MW_pDBrB:.3f} ± {MW_pDBrB_mix2_95CI:.3f} g/mol")
print("Actual Molar Mass: 235.906 g/mol")

:::{admonition} Key Points
- Use data slicing and booleans to find relevant information in an array.
- Plot and analyze a derivative function interactively.
- Compute the molar mass using cooling curves and freezing point depression.
- Use Python functions to save time in doing repetitive tasks.
:::